# Direct Weights Workflow

Use this when a notebook is generating portfolio weights directly instead of generating forecasts first.

## Running This Notebook In Colab

If you want to run this notebook in Google Colab, start by cloning the repo into the Colab session and installing the toolkit in editable mode.

Steps:

1. Set `REPO_URL` below to your GitHub repo URL.
2. Run the bootstrap cell once.
3. After that, the rest of the notebook can import `portfolio_toolkit` normally.

If you are running locally, the same cell will automatically fall back to the repository on your machine.


In [ ]:
# Colab / local bootstrap cell
# - In Colab: clone the repo, install the package, and point repo_root at /content/...
# - Locally: just point repo_root at this repository on disk

import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = 'https://github.com/<your-user-or-org>/<your-repo>.git'  # replace with your real repo URL
    REPO_DIR = '/content/Portfolio-Optimizer'

    if '<your-user-or-org>' in REPO_URL or '<your-repo>' in REPO_URL:
        raise ValueError('Set REPO_URL to your real GitHub repository URL before running this cell in Colab.')

    !rm -rf "$REPO_DIR"
    !git clone "$REPO_URL" "$REPO_DIR"
    %cd "$REPO_DIR"
    %pip install -e ".[dev]"

    repo_root = Path(REPO_DIR).resolve()
else:
    repo_root = Path(repo_root).resolve() if 'repo_root' in globals() else Path('../../').resolve()

print('repo_root =', repo_root)


In [ ]:
from pathlib import Path
import pandas as pd

from portfolio_toolkit import PortfolioWeights, backtest_weights, baseline_weights, start_run, validate_weights_frame, write_backtest_artifacts, log_backtest

repo_root = Path(repo_root).resolve() if 'repo_root' in globals() else Path('../../').resolve()
dataset_name = 'shared_set_1'
output_dir = repo_root / 'runs' / 'direct_weights_workflow'
output_dir.mkdir(parents=True, exist_ok=True)

# Replace this with your own notebook-generated weights.
weights = baseline_weights(dataset_name, 'equal_weight', repo_root=repo_root)
validated_weights = validate_weights_frame(weights.weights, dataset_name=dataset_name, repo_root=repo_root)
portfolio = PortfolioWeights(validated_weights, dataset_name=dataset_name, strategy_name='direct_weights_model')

In [ ]:
result = backtest_weights(dataset_name, portfolio, repo_root=repo_root)
artifact_paths = write_backtest_artifacts(result, output_dir)

with start_run('direct_weights_model', dataset_name, tags={'workflow': 'direct_weights'}, repo_root=repo_root):
    import mlflow
    mlflow.log_params({'split': 'test', 'cost_bps': 10.0, 'source': 'direct_weights'})
    log_backtest(result)

result.metrics

In [ ]:
# Validation cell
assert Path(artifact_paths['quantstats_report']).exists()
assert abs(result.weights.sum(axis=1).iloc[0] - 1.0) < 1e-9
print('Direct weights workflow validated.')